# Create Dave config

Builds the Dave experiment recipe XML from `round_info.csv` (notebook 03) and the Kilroy config for `MICROSCOPE`.

In [ ]:
import os
import sys
from pathlib import Path
import pandas as pd

MERCI_DIR  = Path(os.getcwd()).parent.parent.parent.parent  # MERci/ (notebook lives in MERci/notebooks/prepare_imaging/<variant>/<acquisition>/)
SAMPLE_DIR = MERCI_DIR.parent                  # experiment root, e.g. LT048_sample_18/
sys.path.insert(0, str(MERCI_DIR / "src"))

from MERci.common.experiment_info import resolve_sample_identity
from MERci.acquisition.dave   import create_dave_config, dave_config_filename
from MERci.acquisition.kilroy import find_kilroy_config

In [ ]:
SETTINGS_DIR  = SAMPLE_DIR / "settings"
METADATA_DIR  = SAMPLE_DIR / "metadata"
POSITIONS_DIR = SAMPLE_DIR / "positions"

# SAMPLE_NAME is the TRUE top-level experiment id -- must match what notebooks
# 02/03 used, since the positions files and round_info rows below are named
# with it (see notebook 02's docstring for why this isn't SAMPLE_DIR.name).
SAMPLE_NAME, IMAGING_DIR = resolve_sample_identity(MERCI_DIR)
print(f"SAMPLE_DIR   : {SAMPLE_DIR}")
print(f"SAMPLE_NAME  : {SAMPLE_NAME}")

In [ ]:
# ── Experiment parameters ──────────────────────────────────────────
MICROSCOPE           = "MF4"   # must match what you set in notebooks 01/03
USE_ADAPTORS         = True    # True = adaptor-based fluidics; False = direct readouts
INCLUDE_FINAL_CLEAVE = False   # True = add a final cleave step after last imaging round
FIRST_HYB_NO_CLEAVE  = True    # True = first hyb (after the cells round) omits the cleave step

# Default round structure: imaging round 1 = cells only; rounds 2..N_HYBS+1 = bits #1..#N.
# The fluidics before the first bits round has no cleave; later fluidics include the cleave.

# ── Read what notebook 03 produced ──────────────────────────────────
round_info_path = METADATA_DIR / "round_info.csv"
if not round_info_path.exists():
    raise FileNotFoundError(f"{round_info_path} not found -- run notebook 03 first.")
round_info = pd.read_csv(round_info_path)

rbc_path = METADATA_DIR / "round_bit_color_map.csv"
if not rbc_path.exists():
    raise FileNotFoundError(f"{rbc_path} not found -- run notebook 03 first.")
N_HYBS = int(pd.read_csv(rbc_path)["round"].max())

MULTI_BOUNDARY = "positions_file" in round_info.columns
print(f"N_HYBS               : {N_HYBS}")
print(f"Multi-boundary layout: {MULTI_BOUNDARY}")
print(f"Use adaptors         : {USE_ADAPTORS}")
print(f"Final cleave         : {INCLUDE_FINAL_CLEAVE}")
print(f"First hyb no cleave  : {FIRST_HYB_NO_CLEAVE}")

In [ ]:
# ── Resolve positions inputs for the recipe ──────────────────────────────
if MULTI_BOUNDARY:
    # Per-segment: each round_info row names its own positions file in POSITIONS_DIR.
    positions_arg     = None
    positions_dir_arg = POSITIONS_DIR
    missing = [f for f in round_info["positions_file"].unique()
               if not (POSITIONS_DIR / f).exists()]
    if missing:
        raise FileNotFoundError(
            f"Positions files referenced by round_info are missing (run notebook 02): {missing}"
        )
else:
    # Single-positions: one file for every movie.
    positions_arg     = POSITIONS_DIR / f"positions_{SAMPLE_NAME}.txt"
    positions_dir_arg = None
    if not positions_arg.exists():
        raise FileNotFoundError(f"Positions file not found: {positions_arg}")

# Resolve the Kilroy config that will run this experiment. Its protocol names are
# the source of truth for the fluidic steps written into the Dave recipe, so every
# protocol referenced is guaranteed to exist in Kilroy. If the microscope has no
# Kilroy config, fall back to MF2's.
KILROY_DIR    = MERCI_DIR / "data" / "configs" / "kilroy"
KILROY_CONFIG = find_kilroy_config(MICROSCOPE, KILROY_DIR, fallback_microscope="MF2")
print(f"Kilroy config (protocol source): {KILROY_CONFIG.name}")

dave_output = SETTINGS_DIR / dave_config_filename(MICROSCOPE, N_HYBS, SAMPLE_NAME)

create_dave_config(
    round_info           = round_info,
    positions_file       = positions_arg,
    settings_dir         = SETTINGS_DIR,
    output_path          = dave_output,
    use_adaptors         = USE_ADAPTORS,
    include_final_cleave = INCLUDE_FINAL_CLEAVE,
    first_hyb_no_cleave  = FIRST_HYB_NO_CLEAVE,
    kilroy_config        = KILROY_CONFIG,
    positions_dir        = positions_dir_arg,
)

print(f"Dave config saved: {dave_output}")

# Preview the generated file
with open(dave_output, encoding="ISO-8859-1") as fh:
    print(fh.read())

## Kilroy config

Locates the Kilroy config for the current microscope in `MERci/data/configs/kilroy/`
(newest matching file by `YYMMDD` date stamp) and copies it to `SAMPLE_DIR/settings/`.
This is the same file used above as the protocol source for the Dave recipe. If the
microscope has no Kilroy config, it falls back to MF2's.

In [7]:
import shutil

# Same resolution (with MF2 fallback) used as the Dave protocol source above.
kilroy_src  = find_kilroy_config(MICROSCOPE, MERCI_DIR / "data" / "configs" / "kilroy",
                                 fallback_microscope="MF2")
kilroy_dest = SETTINGS_DIR / kilroy_src.name
shutil.copy2(str(kilroy_src), str(kilroy_dest))
print(f"Kilroy config  : {kilroy_src.name}")
print(f"Copied to      : {kilroy_dest}")

Kilroy config  : kilroy-config-mf4-direct-and-adaptors-260617.xml
Copied to      : c:\Users\Leonardo\Dropbox\research\analysis\LineageTracing\251225_LT027_saving_time\settings\kilroy-config-mf4-direct-and-adaptors-260617.xml
